## 3.1 Analyzing Token Frequency
- Goal: Determine the minimum threshold

In [1]:
import pandas as pd
from collections import Counter
train_v2_token = pd.read_parquet('data/processed/train_v2_tokenized.parquet')

token_frequency = Counter()
for tokens in train_v2_token['tokens']:
    token_frequency.update(tokens)


filtered_counter = {word: count for word, count in token_frequency.items() if count <= 4}
print(filtered_counter)
print(f"Length of Tokens: {len(token_frequency)}")
print(f"Length of Filtered Tokens: {len(filtered_counter)}")


{'drad': 2, 'hrad': 4, 'mrad': 4, 'nsl': 2, "'{:03}'": 2, 'twoleft': 2, 'agg': 2, 'eat_c': 2, 'ecnt': 2, 'acr': 2, 'bcr': 2, 'num_max': 2, 'num_yoko': 2, 'num_tate': 2, 'Y_MAX': 2, 'NN': 4, 'len_A': 2, '"AAAAAAAAAA"': 1, 'additional': 1, 'RL': 2, 'sort_v': 1, 'code': 3, 'must_do_a': 2, 'like2017': 1, 'sprit': 1, 'next_nodes': 2, 'edge_dict': 2, 'top_keta': 1, 'theta_2': 2, '36': 2, 'card_type': 2, 'card_num': 4, 'sympy': 1, 'nlis': 1, 'ac_list': 1, 'like_2017': 4, 'b100': 4, 'fin': 1, 'outPut2': 1, 'attack': 3, 'new_div': 1, 'CB': 1, 'capable': 1, "'\\.\\.\\.'": 2, 'aup': 2, 'li_': 1, 'ABS': 2, 'elems': 2, 'Ansn': 2, 'g_s': 4, 'like_num': 1, 'median_left': 4, 'median_right': 2, 'c_dash': 2, 'al_set': 2, 'sevens': 1, 'result_list': 3, 'alphabet': 2, 'good_dfs': 2, 'gi': 2, 'sdis': 2, 'colections': 1, 'ea': 4, 'easb': 3, 'average': 4, '"0 "': 2, 'tempstring': 4, '"MMYY"': 4, '"YYMM"': 2, 'taps': 1, 'sosuu': 2, 'modlist': 1, 'nr': 1, 'nq': 3, 'matha': 1, 'S_s': 1, 'dict_': 3, 'sou': 1, 'a

## Investigating Rows with Comments
- Some of the rows had comments so in order to decide on how to move forward (remove or keep), I need to look at how important these comments are to the code
- Looked at rows with only comments to see if they were used for lables to determine if we can remove the comment tokens
- Conclusion: None of the rows in the examples were made up purely of just comments. The comment tokens that were found contribute to over 1,600 occurances consisting primarily of debug statements, notebook artfacts, encoding declarations, and natural language comments. Manual inspection showed that the five target labels were determined by the changes to the executable Python code rather than the comment contents. Comment tokens were tremoved from the tokenizer to reduce vocabulary size, while maintaining the represenation of executable code for the label.

In [2]:
from src.v2.comment_only_diff import is_comment_only

filtered_contains_comments = train_v2_token[train_v2_token['diff'].str.contains('#')]
print(f'Amount of rows that contain comments: {len(filtered_contains_comments)}')
for i in [1, 2, 3, 10, 20, 30, 50, 100, 150, 200, 300]:
    print(f"{i} ---- {filtered_contains_comments['diff'].iloc[i]} \n Top Level Label: {filtered_contains_comments['top_level_label'].iloc[i]}")

#Goal: return all of the lines that have comments only so this returns true
filtered_only_comments = train_v2_token.copy()
filtered_only_comments['only_comments'] = train_v2_token['diff'].apply(is_comment_only)
display(filtered_only_comments[filtered_only_comments['only_comments'] == True])

#Goal: find all of the unique comments strings in diff
unique_comments = Counter()
for row in filtered_contains_comments['diff']:
    lines = row.splitlines()
    for line in lines:
        line = line.replace(line[0], '', 1)
        line = line.strip()
        if not line:
            continue
        elif line.startswith('#'):
            unique_comments[line] += 1

for comment in unique_comments.most_common(10):
    print(comment)

print(len(unique_comments))
    

Amount of rows that contain comments: 1471
1 ---- - print(a)
+ #print(a) 
 Top Level Label: call
2 ---- - print(t)
+ # print(t) 
 Top Level Label: call
3 ---- - print(dp)
+ #print(dp) 
 Top Level Label: call
10 ---- + #162a
-     print(top_keta)
+     #print(top_keta) 
 Top Level Label: call
20 ---- - 		print(i)
+ 		#print(i) 
 Top Level Label: call
30 ---- -         print(sum_list)
+         # print(sum_list) 
 Top Level Label: call
50 ---- -       if s[i - 1] == s[j - 1] and LCSRe[i - 1][j - 1] < (j - i): 
+       if s[i - 1] == s[j - 1] and LCSRe[i - 1][j - 1] < j - i: 
-         else: 
+       else: 
-           LCSRe[i][j] = 0
+         LCSRe[i][j] = 0
+   #print(LCSRe) 
 Top Level Label: control_flow
100 ---- -     print(v)
+ # print(f"{pi} {qi}")
+ 
+  
 Top Level Label: call
150 ---- -   print(s)
+   #print(s) 
 Top Level Label: call
200 ---- - print(ans)
+ #print(ans) 
 Top Level Label: call
300 ---- - print(l)
+ #print(l) 
 Top Level Label: call


,diff,top_level_label,tokens,only_comments


('#print(s)', 30)
('#print(i)', 29)
('#print(a)', 20)
('#print(ans)', 19)
('#print(l)', 18)
('#print(dp)', 16)
("# '11111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111

## Investigating Various Rows For Determining Minimum Threshold
- I saw tokens like `10000000000000` and I wanted to see it in context
- Upon inspecting an example, I found a top label of `call` where it had lots of indendation surrounding it. This could be potential noise for the model so I decided to change the tokenizer to reflect the indentation rather than just removing it.

In [4]:
contains_token = train_v2_token[train_v2_token['diff'].str.contains('gcds')]

print(len(contains_token['diff']))
for i in range(1):
    print(f'Top Level Label: {contains_token['top_level_label'].iloc[i]}')
    print(contains_token['diff'].iloc[i])
    print('------')


1
Top Level Label: expression
+ 
- for i in range(n + 1):
+ for i in range(n):
- 	gcds = max(gcds,(math.gcd(r[i],l[n - i])))
+ 	gcds = max(gcds,(math.gcd(r[i],l[n - i - 1])))
------
